In [1]:
from jax import jit, vmap
import jax.numpy as jnp
from jax.lax import cond, while_loop
import jax.random as jr
import numpy as np

In [2]:
@jit
def searchsorted_2D(vector: jnp.ndarray, matrix: jnp.ndarray) -> jnp.ndarray:
	"""
	Search along axis 1 for a vector in a matrix. If found, return the index of the vector.
	If not found, return len(matrix).

	For this function to work, the vectors in the matrix must be sorted lexicographically.
	ex:
	[[1, 1, 0],
	 [1, 2, 1],
	 [1, 2, 2],
	 [2, 1, 3],
	 [2, 2, 1]]

	:param vector: the vector to search for
	:param matrix: the matrix to search in
	:return: the index of the vector in the matrix, or len(matrix) if not found
	"""

	@jit
	def compare_vectors(v1, v2):
		"""Compare two vectors lexicographically. Returns -1 if v1 < v2, 0 if equal, 1 if v1 > v2"""
		diff = v1 - v2
		# Find first non-zero element
		nonzero_mask = diff != 0
		# If all elements are zero, vectors are equal
		first_nonzero_idx = jnp.argmax(nonzero_mask)

		return cond(
			jnp.any(nonzero_mask),
			lambda: jnp.array(jnp.sign(diff[first_nonzero_idx])).astype(int),
			lambda: jnp.array(0).astype(int)
		)

	@jit
	def search_condition(state):
		start, end, found = state
		return (start < end) & (~found)

	@jit
	def search_step(state):
		start, end, found = state
		mid = (start + end) // 2

		comparison = compare_vectors(vector, matrix[mid])

		# If vectors are equal, we found it
		new_found = comparison == 0
		new_start = cond(comparison < 0, lambda: start, lambda: mid + 1)
		new_end = cond(comparison < 0, lambda: mid, lambda: end)

		# If found, return the index in start position
		final_start = cond(new_found, lambda: mid, lambda: new_start)

		return final_start, new_end, new_found

	# Initial state: (start, end, found)
	initial_state = (0, len(matrix), False)
	final_start, final_end, found = while_loop(search_condition, search_step, initial_state)

	# Return the found index or len(matrix) if not found
	return cond(found, lambda: final_start, lambda: len(matrix))


searchsorted_2D_vectorized = jit(vmap(searchsorted_2D, in_axes=(0, None)))


@jit
def lexicographic_sort(arr: jnp.ndarray) -> jnp.ndarray:
	"""
	sorts a 2D array lexicographically
	:param arr: 2D array to be sorted
	:return: sorted version along the first dimension
	"""
	return arr[jnp.lexsort(arr.T[::-1])]


@jit
def compute_mapping(grid: jnp.array, element: jnp.array) -> jnp.array:
	"""
	Returns the indices of each vector/element in the grid. The grid must be sorted lexicographically.

	:param grid: a sorted grid of shape (N,) or (N, I). If it is 2D, it must be sorted lexicographically.
	:param elements: element to search for in the grid, either scalar or of shape (I,)
	:return: indices in grid where the element can be found, as a scalar.
	"""
	if grid.shape[-1] == 1:
		# We only have 1 input dimension, and we can use the fast jnp.searchsorted function
		return jnp.searchsorted(grid.squeeze(axis=-1), element.squeeze(axis=-1))
	# Multiple input dimensions requires our custom lexicographic search
	return searchsorted_2D_vectorized(element, grid)

In [3]:
G = 1500
I = 2
T = 1000
N = 500

key = jr.PRNGKey(42)

In [4]:
# generates a random G*I grid
key, subkey = jr.split(key)
grid = jr.uniform(subkey, (G, I))

In [5]:
# choose T series of N elements in the grid
key, subkey = jr.split(key)
subkeys = jr.split(subkey, T)
tasks = vmap(lambda k: grid[jr.choice(k, G, (N,), replace=False)], in_axes=0)(subkeys)

In [6]:
np.asarray(grid.T)

array([[0.72766423, 0.18169427, 0.11072934, ..., 0.06842351, 0.01958179,
        0.48926628],
       [0.78786755, 0.26263022, 0.20263076, ..., 0.08957648, 0.57570004,
        0.7428132 ]], shape=(2, 1500), dtype=float32)

In [7]:
np.asarray(tasks[0].T)

array([[1.12536073e-01, 2.56549001e-01, 8.23353648e-01, 8.90067697e-01,
        8.95409107e-01, 7.21436739e-02, 4.18542862e-01, 8.32777023e-02,
        6.01688743e-01, 8.53570104e-01, 8.48481297e-01, 6.40035868e-01,
        6.33792639e-01, 2.94964314e-02, 6.12479329e-01, 2.39347219e-01,
        9.67597127e-01, 2.95840144e-01, 1.79924250e-01, 3.43882799e-01,
        2.23622680e-01, 4.37557101e-01, 3.88570309e-01, 1.14853382e-02,
        9.68735695e-01, 4.77303982e-01, 7.25711346e-01, 9.36998010e-01,
        4.92689252e-01, 5.29391766e-02, 6.81283116e-01, 2.39783645e-01,
        3.26254487e-01, 2.34465361e-01, 7.16232181e-01, 2.96609402e-02,
        9.98437762e-01, 6.05338812e-01, 9.95449901e-01, 4.23371196e-01,
        1.10181570e-01, 8.21169019e-01, 5.87427139e-01, 2.43086815e-02,
        7.00384021e-01, 8.87311459e-01, 9.14162755e-01, 6.67189598e-01,
        8.47614646e-01, 4.64523077e-01, 6.29744649e-01, 2.68926263e-01,
        2.25179315e-01, 4.61060405e-01, 6.66268229e-01, 9.782843

In [8]:
np.asarray(lexicographic_sort(grid).T)

array([[4.6622753e-04, 5.2928925e-04, 2.1110773e-03, ..., 9.9860466e-01,
        9.9921942e-01, 9.9969220e-01],
       [9.3930459e-01, 9.4958055e-01, 6.0286045e-01, ..., 4.4553649e-01,
        2.0154905e-01, 8.7344098e-01]], shape=(2, 1500), dtype=float32)

In [9]:
@jit
def mappings_old(grid, tasks):
	sorted_grid = lexicographic_sort(grid)

	return vmap(lambda t: compute_mapping(sorted_grid, t))(tasks)

In [10]:
mappings_old(grid, tasks)

Array([[ 184,  403, 1223, ..., 1403,  344, 1358],
       [1278,  959,  742, ..., 1064,  377,  890],
       [ 971, 1228,  535, ..., 1400, 1436, 1172],
       ...,
       [ 360,  897,   43, ...,   46,  912,  332],
       [ 974,  953, 1254, ...,  801, 1175, 1005],
       [1194,  577,  448, ..., 1005,  798,  669]],      dtype=int32, weak_type=True)

In [11]:
%timeit -n 10 -r 3 mappings_old(grid, tasks).block_until_ready()

364 μs ± 7.63 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [12]:
s_grid = lexicographic_sort(grid)

In [13]:
dists = jnp.linalg.norm(s_grid[:, None, :] - tasks[0][None, :, :], axis=-1)
index = jnp.argmin(dists, axis=0)
index

Array([ 184,  403, 1223, 1326, 1329,  119,  627,  139,  903, 1275, 1265,
        969,  957,   45,  924,  375, 1444,  459,  287,  531,  352,  653,
        584,   16, 1448,  718, 1100, 1399,  746,   83, 1028,  378,  508,
        371, 1084,   46, 1496,  909, 1491,  637,  180, 1215,  883,   36,
       1058, 1321, 1368, 1010, 1263,  698,  949,  420,  356,  690, 1008,
       1466, 1031, 1352, 1091,  930,  151,  372,  631,  786, 1156, 1240,
        723, 1177,  429,  510,  643,  868,  176,  844,  327,  492, 1465,
        482, 1384, 1256,  737,  188, 1249,  394,   23, 1032,  983,   65,
        264, 1222, 1090,  671,  232,  916,  936, 1036,  749,  575,  488,
        197,  161, 1364,  263,  640, 1330,  189,  290, 1422,  776,  970,
         96,  140,  116,  879,  994, 1123,  239,  313,  122,  101,  977,
        955, 1149, 1197,  428, 1178,  599, 1264, 1157,  495,  369,  927,
        274,  252, 1442, 1118,  765,  390, 1337,  305,    2, 1447,  467,
       1052, 1288, 1306, 1260,  660, 1401, 1165, 13

In [14]:
@jit
def mappings_new(grid, tasks):
	@jit
	def task_mapping(task):
		t = jnp.sum(task**2, axis=-1, keepdims=True)
		g = jnp.sum(grid**2, axis=-1, keepdims=True).T

		dists = t + g - 2 * task @ grid.T
		return jnp.argmin(dists, axis=1)

	return vmap(task_mapping)(tasks)

In [15]:
@jit
def mappings_new(grid, tasks):
	return vmap(lambda t: jnp.argmin(jnp.linalg.norm(grid[:, None, :] - t[None, :, :], axis=-1), axis=0))(tasks)

In [16]:
mappings_new(s_grid, tasks)

Array([[ 184,  403, 1223, ..., 1403,  344, 1358],
       [1278,  959,  742, ..., 1064,  377,  890],
       [ 971, 1228,  535, ..., 1400, 1436, 1172],
       ...,
       [ 360,  897,   43, ...,   46,  912,  332],
       [ 974,  953, 1254, ...,  801, 1175, 1005],
       [1194,  577,  448, ..., 1005,  798,  669]], dtype=int32)

In [17]:
%timeit -n 10 -r 3 mappings_new(s_grid, tasks).block_until_ready()

363 μs ± 5.33 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [18]:
assert jnp.all(mappings_old(grid, tasks) == mappings_new(s_grid, tasks))